# M2 Notebook Release Header
**Release Header (Auto)**
- Purpose: analysis workflow notebook.
- Inputs: local source tables and matrices.
- Outputs: figures and summary metrics.
- Dependencies: existing scientific Python packages only.
- Execution Order: run top-to-bottom.


# Publish Notebook Header

- Purpose: Reproducible analysis notebook for M2 release package.
- Inputs: Local project data files (configured via relative paths or project root variable).
- Outputs: Analysis tables and figures with unchanged logic/style.
- Execution Order: Run cells top-to-bottom.
- Privacy: Local identity/path tokens are anonymized for release.


-PCAmotion -


## Section: Core Analysis
This section preserves original computation and visualization behavior.


In [56]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from scipy.spatial import ConvexHull
import plotly.express as px
import os

In [57]:
def PCA_analysis(data, im_data, csv_path):
    # Standardize the data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Perform PCA
    pca = PCA(n_components=2)
    data_pca = pca.fit_transform(data_scaled)

    maxlen = min(data.shape[0],im_data.shape[0])
    if data.shape[0]>maxlen:
        data = data.iloc[:maxlen,:]
    if im_data.shape[0]>maxlen:
        im_data = im_data.iloc[:maxlen,:]

    pca = PCA(n_components=2)
    data_pca = pca.fit_transform(data.to_numpy())
    explained_variance = pca.explained_variance_ratio_
    print(f'Explained variance: {explained_variance}')

    time_axis = np.arange(data_pca.shape[0]).reshape(-1, 1)
    data_3d = np.hstack((data_pca, time_axis))
    data_3d = data_3d.astype(np.float64)
    data_3d = pd.DataFrame(data_3d, columns=['PCA1', 'PCA2', 'time'])
    data_4d = pd.concat([data_3d, im_data], axis=1)
    data_4d.to_csv(csv_path, index=False)

    return data_4d

In [58]:
def PCA_plot(data_4d,title_prefix,fig_path,is_save=False):
    #interactable 3-d plot
    # fig = px.scatter_3d(data_4d, x='PCA1', y='PCA2', z='time',
    #                     color='processed_state',
    #                     title='3D PCA Motion'
    #                     )
    # fig.show()
    # plt.close('all')

    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    color_map = {0: 'blue', 1: 'red'}
    # Group by the label column ('processed_state') and plot each group
    for label, group in data_4d.groupby('processed_state'):
        ax.scatter(group['PCA1'], group['PCA2'], group['time'], label=f'Label {label}', alpha=0.5, c=color_map[label])

    # Add axis labels and title
    ax.set_xlabel('PCA1')
    ax.set_ylabel('PCA2')
    ax.set_zlabel('Time')
    ax.set_title(f'{title_prefix} 3D PCA ')
    ax.view_init(azim=65)
    ax.legend(title='IM State')

    if is_save:
        plt.savefig(f'{fig_path}_3D_PCAMotion.pdf', dpi=300)
    plt.show()
    plt.close('all')

    # Create a 2D scatter plot
    fig, ax = plt.subplots(figsize=(10, 8))

    # Define a color map for binary labels (0 -> blue, 1 -> red)
    # Group by the label column ('processed_state') and plot each group
    for label, group in data_4d.groupby('processed_state'):
        ax.scatter(
            group['PCA1'], 
            group['PCA2'], 
            label=f'Label {label}', 
            alpha=0.7, 
            c=color_map[label]
        )
        #plot_confidence_ellipse(group[['PCA1', 'PCA2']].to_numpy(), ax, n_std=2.5, edgecolor=color_map[label][0], facecolor='none', linewidth=2)

    # Add axis labels and title
    ax.set_xlabel('PCA1 ')
    ax.set_ylabel('PCA2 ')
    ax.set_title(f'{title_prefix} 2D PCA Motion')
    ax.legend(title='IM State')

    # Save the plot
    if is_save:
        plt.savefig(f'{fig_path}_2D_PCAMotion.pdf', dpi=300)
    plt.show()
    plt.close('all')

In [59]:
def plot_confidence_ellipse(data, ax, n_std=2, **kwargs):
    """   
    Plot a confidence ellipse for 2D data.
    """
    cov = np.cov(data, rowvar=False)  # Covariance matrix
    mean = np.mean(data, axis=0)     # Mean of the data

    # Compute the eigenvalues and eigenvectors
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Compute the angle and width/height of the ellipse
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigvals)

    # Draw the ellipse
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)
    return ellipse

In [60]:
projectpath = '/home/user_theft/XJN_M2project'
test = 'test2'##VIP:1~4;CamkII_TST:1~10;SST:1~3;PV:1~7;Hsyn:1~5

#test1 params
miceID = range(1,11)#VIP:1~4;CamkII_TST:1~10;SST:1~3;PV:1~7;Hsyn:1~5
cell_types = ['CamkII_TST']

#test2 params
condition = 'rescue'
experiment = 'TST'
align = 'aligned'
mice_IDs = ["LHQ50","LHQ30","LH5799","LH5798","LH0166","LH0167",
           "CSDS0087","CSDS0126","CSDS0179",
           "CSDSQ29", "CSDSQ27", "CSDSQ26", "CSDS5797", "CSDS5776","LH5779","CSDS0103","CSDS0370"]
#"LH5779","CSDS0103","CSDS0370" no pre behaviroal

In [ ]:
for mice in mice_IDs:
        if test == 'test1':
            print('test1 mode, skipping this looper')
            break

        signal_path = os.path.join(projectpath, 'signal_data', test, condition, mice,'signal_save',f'{mice}_{condition}_{align}_table_{experiment}.csv')
        im_path = os.path.join(projectpath, 'behavioral_data', test, condition, mice, 'im_preprocessed.csv')
        out_path = os.path.join(projectpath, 'summary', test, condition, mice,experiment, 'PCA_Analysis')

        if not os.path.exists(out_path):os.makedirs(out_path)

        #check wired cases
        if not os.path.exists(signal_path) and not os.path.exists(im_path):
            print(f'No data for {mice}{condition}, skipping...')
            continue
        elif not os.path.exists(signal_path):
            print(f'No signal data for {signal_path}')
            continue 
        elif not os.path.exists(im_path):
            raise FileNotFoundError(f'No behavioral data for {im_path}')
        
        signal_data = pd.read_csv(signal_path)
        im_data = pd.read_csv(im_path)
        data_4d = PCA_analysis(signal_data, im_data, os.path.join(out_path, f'{mice}{condition}_PCAComponents.csv'))
        PCA_plot(data_4d, f'{mice}{condition}_{experiment}', os.path.join(out_path, f'{mice}{condition}'), is_save=True)
        
        

